# Week 3, days 1–4 on a Mac: HuggingFace without Colab

The course runs days 1–4 in Colab on a T4. This notebook redoes the same three labs locally,
on Apple Silicon (`mps`), with small **ungated** models so nothing needs a HuggingFace token:

1. **Day 2 – pipelines**: the high-level API
2. **Day 3 – tokenizers**: how text turns into tokens, and why Spanish costs more than English
3. **Day 4 – models**: the low-level API – load a model, apply a chat template, generate and stream

Day 1 is the Colab introduction itself, so there is no code for it here.
The first run downloads about 1.5 GB of models into `~/.cache/huggingface`.

In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TextStreamer

device = "mps" if torch.backends.mps.is_available() else "cpu"
CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # ~1 GB, ungated, speaks Spanish
print(f"torch {torch.__version__} on {device}")

torch 2.9.0 on mps


## Day 2 – Pipelines

One line per task: the pipeline picks a tokenizer, a model and the pre/post-processing.

In [2]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=device)
classifier(["I finally understood the subjunctive today!", "Irregular verbs are ruining my week."])

[{'label': 'POSITIVE', 'score': 0.9964125752449036},
 {'label': 'NEGATIVE', 'score': 0.9997735619544983}]

In [3]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)
ner("Ana moved from Madrid to Buenos Aires to teach Spanish at the University of Buenos Aires.")

[{'entity_group': 'PER',
  'score': np.float32(0.9982961),
  'word': 'Ana',
  'start': 0,
  'end': 3},
 {'entity_group': 'LOC',
  'score': np.float32(0.9995921),
  'word': 'Madrid',
  'start': 15,
  'end': 21},
 {'entity_group': 'LOC',
  'score': np.float32(0.9995655),
  'word': 'Buenos Aires',
  'start': 25,
  'end': 37},
 {'entity_group': 'MISC',
  'score': np.float32(0.99934965),
  'word': 'Spanish',
  'start': 47,
  'end': 54},
 {'entity_group': 'ORG',
  'score': np.float32(0.98798543),
  'word': 'University of Buenos',
  'start': 62,
  'end': 82},
 {'entity_group': 'LOC',
  'score': np.float32(0.8486237),
  'word': 'Aires',
  'start': 83,
  'end': 88}]

In [4]:
fill = pipeline("fill-mask", model="distilbert/distilroberta-base", device=device)
[(r["token_str"], round(r["score"], 3)) for r in fill("The best way to learn a language is to <mask> every day.")]

[(' train', 0.168),
 (' practice', 0.167),
 (' learn', 0.106),
 (' write', 0.083),
 (' work', 0.071)]

In [5]:
generator = pipeline("text-generation", model=CHAT_MODEL, device=device)
messages = [{"role": "user", "content": "Translate into Spanish: 'Where is the train station?' Reply with the translation only."}]
generator(messages, max_new_tokens=40, do_sample=False)[0]["generated_text"][-1]["content"]

'¿Dónde está la estación de tren?'

## Day 3 – Tokenizers

The same sentence costs a different number of tokens in each tokenizer, and usually more in Spanish than in English.

In [6]:
tokenizers = {
    "gpt2": AutoTokenizer.from_pretrained("openai-community/gpt2"),
    "qwen2.5": AutoTokenizer.from_pretrained(CHAT_MODEL),
    "phi-3": AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct"),
}
english = "I would like to order a coffee with milk and a croissant, please."
spanish = "Me gustaría pedir un café con leche y un cruasán, por favor."

for name, tok in tokenizers.items():
    print(f"{name:8} vocab={len(tok):>7,}  english={len(tok.encode(english, add_special_tokens=False)):>3}  spanish={len(tok.encode(spanish, add_special_tokens=False)):>3}")

gpt2     vocab= 50,257  english= 17  spanish= 20
qwen2.5  vocab=151,665  english= 16  spanish= 19
phi-3    vocab= 32,011  english= 16  spanish= 21


In [7]:
tok = tokenizers["qwen2.5"]
ids = tok.encode(spanish, add_special_tokens=False)
print(ids)
print(tok.convert_ids_to_tokens(ids))  # Ġ marks a leading space
print(tok.decode(ids))

[7823, 29814, 82701, 10477, 404, 650, 51950, 390, 512, 1528, 379, 650, 22218, 300, 11390, 11, 4154, 4694, 13]
['Me', 'Ġgust', 'arÃŃa', 'Ġped', 'ir', 'Ġun', 'ĠcafÃ©', 'Ġcon', 'Ġle', 'che', 'Ġy', 'Ġun', 'Ġcru', 'as', 'Ã¡n', ',', 'Ġpor', 'Ġfavor', '.']
Me gustaría pedir un café con leche y un cruasán, por favor.


Instruct models also expect a **chat template**: special tokens that mark who is speaking.

In [8]:
chat = [
    {"role": "system", "content": "You are a patient Spanish tutor."},
    {"role": "user", "content": "How do I say 'I am tired'?"},
]
print(tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True))

<|im_start|>system
You are a patient Spanish tutor.<|im_end|>
<|im_start|>user
How do I say 'I am tired'?<|im_end|>
<|im_start|>assistant



## Day 4 – Models

Now without pipelines: tokenizer → tensors → `model.generate` → decode.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, torch_dtype=torch.float16).to(device)
print(f"{model.num_parameters():,} parameters, {model.get_memory_footprint() / 1e9:.2f} GB")
model

494,032,768 parameters, 0.99 GB


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [10]:
def generate(messages, max_new_tokens=200):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    outputs = model.generate(inputs, max_new_tokens=max_new_tokens, streamer=streamer, do_sample=False)
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

reply = generate([
    {"role": "system", "content": "You are a Spanish tutor. Be brief."},
    {"role": "user", "content": "Explain the difference between 'ser' and 'estar' with two examples each."},
])

**Ser** 

is 

used 

to 

describe 

a 

... (53 outputs left out)

leyendo." 

(I 

am 

reading.)


In [11]:
# Free memory before moving on
del model, generator
if device == "mps":
    torch.mps.empty_cache()